# 📦 TMS2 - Create Unified Dataset & Upload to Kaggle

This notebook processes all your traffic datasets into a unified YOLO-format dataset and uploads it to Kaggle.

### Datasets to Process:
| Dataset | License | Included |
|---------|---------|----------|
| UA-DETRAC | CC0/CC BY 4.0 | ✅ |
| Real-Time Traffic | Open | ✅ |
| Serbia Traffic | Open | ✅ |
| Traffic Density Singapore | Open | ✅ |
| LISA Traffic Light | CC BY-NC-SA 4.0 | ✅ |

> ⚠️ **License Note:** Due to LISA's CC BY-NC-SA 4.0 license, this combined dataset is **NON-COMMERCIAL**.

---

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/tms2_colab_training'
DATA_PATH = f'{DRIVE_PATH}/data'
KAGGLE_PATH = f'{DATA_PATH}/kaggle'
OUTPUT_PATH = f'{DATA_PATH}/tms2_unified_dataset'

import os
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(f'{OUTPUT_PATH}/images/train', exist_ok=True)
os.makedirs(f'{OUTPUT_PATH}/images/val', exist_ok=True)
os.makedirs(f'{OUTPUT_PATH}/images/test', exist_ok=True)
os.makedirs(f'{OUTPUT_PATH}/labels/train', exist_ok=True)
os.makedirs(f'{OUTPUT_PATH}/labels/val', exist_ok=True)
os.makedirs(f'{OUTPUT_PATH}/labels/test', exist_ok=True)

print(f"Output: {OUTPUT_PATH}")

Mounted at /content/drive
Output: /content/drive/MyDrive/tms2_colab_training/data/tms2_unified_dataset


In [2]:
!pip install -q ultralytics opencv-python kaggle pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 31.7 MB/s eta 0:00:00


In [3]:
import cv2
import numpy as np
import pandas as pd
import glob
import shutil
import json
import random
from pathlib import Path
from tqdm import tqdm
from datetime import datetime

print("Libraries loaded!")

Libraries loaded!


## 2. ⚙️ CONFIGURATION

In [4]:
# ============================================================
# ⚙️ CONFIGURATION - EDIT THESE VALUES
# ============================================================

# === KAGGLE SETTINGS ===
KAGGLE_USERNAME = 'kunlaisnghh25'  # Your Kaggle username
DATASET_SLUG = 'tms2-traffic-unified'
DATASET_TITLE = 'TMS2 Unified Traffic Dataset'

# === PROCESSING LIMITS ===
MAX_IMAGES_PER_DATASET = None     # Max images from each source
MAX_FRAMES_PER_VIDEO = None        # Max frames to extract per video
FRAME_SAMPLE_INTERVAL = 5        # Extract every Nth frame

# === SPLIT RATIOS ===
TRAIN_RATIO = 0.7
VAL_RATIO = 0.2
TEST_RATIO = 0.1  # Remaining

# === DATASETS TO INCLUDE ===
# All datasets included - dataset will be CC BY-NC-SA 4.0 due to LISA
INCLUDE_DATASETS = {
    'ua-detrac': True,              # CC0/CC BY 4.0
    'real-time-traffic': True,      # Open
    'serbia-traffic': True,         # Open
    'traffic-density-singapore': True,  # Open
    'lisa-traffic-light': True,     # ✅ CC BY-NC-SA 4.0 (Non-Commercial)
}

# === CLASS MAPPING ===
# Unified classes for all datasets
CLASSES = ['car', 'motorcycle', 'bus', 'truck', 'traffic_light']
COCO_TO_UNIFIED = {2: 0, 3: 1, 5: 2, 7: 3, 9: 4}  # COCO class IDs to our IDs

print("📊 Configuration:")
print(f"  Kaggle: {KAGGLE_USERNAME}/{DATASET_SLUG}")
print(f"  Max images/dataset: {MAX_IMAGES_PER_DATASET}")
print(f"  Split: {TRAIN_RATIO:.0%} train / {VAL_RATIO:.0%} val / {TEST_RATIO:.0%} test")
print("\n  ⚠️ License: CC BY-NC-SA 4.0 (Non-Commercial due to LISA)")
print("\n  Included datasets:")
for name, include in INCLUDE_DATASETS.items():
    print(f"    {'✅' if include else '❌'} {name}")

📊 Configuration:
  Kaggle: kunlaisnghh25/tms2-traffic-unified
  Max images/dataset: None
  Split: 70% train / 20% val / 10% test

  ⚠️ License: CC BY-NC-SA 4.0 (Non-Commercial due to LISA)

  Included datasets:
    ✅ ua-detrac
    ✅ real-time-traffic
    ✅ serbia-traffic
    ✅ traffic-density-singapore
    ✅ lisa-traffic-light


## 3. Discover Source Datasets

In [6]:
# Dataset paths and configurations
DATASET_CONFIGS = {
    'ua-detrac': {
        'paths': [f'{KAGGLE_PATH}/UA_DETRAC', f'{KAGGLE_PATH}/UA_DETRAC'],
        'has_annotations': True,
        'annotation_format': 'yolo',
    },
    'real-time-traffic': {
        'paths': [f'{KAGGLE_PATH}/RealTime_Traffic_Videos'],
        'has_annotations': True,
        'annotation_format': 'yolo',
    },
    'serbia-traffic': {
        'paths': [f'{KAGGLE_PATH}/RoadTraffic_Serbia'],
        'has_annotations': True,
        'annotation_format': 'yolo',
    },
    'traffic-density-singapore': {
        'paths': [f'{KAGGLE_PATH}/Traffic_Density_Singapore'],
        'has_annotations': False,
        'annotation_format': None,
    },
    'lisa-traffic-light': {
        'paths': [f'{KAGGLE_PATH}/LISA_Traffic_Light'],
        'has_annotations': True,
        'annotation_format': 'csv',
    },
}

# Discover available data
discovered = {}
print("🔍 Discovering datasets...\n")

for name, config in DATASET_CONFIGS.items():
    if not INCLUDE_DATASETS.get(name, False):
        print(f"⏭️ {name}: Excluded")
        continue

    for path in config['paths']:
        if os.path.exists(path):
            images = glob.glob(f'{path}/**/*.jpg', recursive=True)
            images += glob.glob(f'{path}/**/*.png', recursive=True)
            videos = glob.glob(f'{path}/**/*.avi', recursive=True)
            videos = glob.glob(f'{path}/**/*.mov', recursive=True)
            videos = glob.glob(f'{path}/**/*.MOV', recursive=True)
            videos += glob.glob(f'{path}/**/*.mp4', recursive=True)
            labels = glob.glob(f'{path}/**/*.txt', recursive=True)
            csv_files = glob.glob(f'{path}/**/*.csv', recursive=True)

            discovered[name] = {
                'path': path,
                'images': sorted(images),
                'videos': sorted(videos),
                'labels': labels,
                'csv_files': csv_files,
                'has_annotations': len(labels) > 0 or len(csv_files) > 0 or config['has_annotations'],
                'format': config['annotation_format'],
            }
            print(f"✅ {name}: {len(images)} images, {len(videos)} videos")
            break
    else:
        print(f"❌ {name}: Not found")

print(f"\n📊 Found {len(discovered)} datasets")

🔍 Discovering datasets...

✅ ua-detrac: 13452 images, 0 videos
✅ real-time-traffic: 0 images, 2 videos
✅ serbia-traffic: 71 images, 0 videos
✅ traffic-density-singapore: 4027 images, 0 videos
✅ lisa-traffic-light: 44075 images, 0 videos

📊 Found 5 datasets


## 4. Load YOLO Model for Pseudo-Labeling

In [7]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
print("YOLOv8 loaded for pseudo-labeling!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLOv8 loaded for pseudo-labeling!


## 5. Processing Functions

In [8]:
def get_split():
    """Randomly assign train/val/test split."""
    r = random.random()
    if r < TRAIN_RATIO:
        return 'train'
    elif r < TRAIN_RATIO + VAL_RATIO:
        return 'val'
    return 'test'


def find_yolo_label(image_path):
    """Find corresponding YOLO label file for an image."""
    img = Path(image_path)

    label = img.with_suffix('.txt')
    if label.exists():
        return label

    label = img.parent.parent / 'labels' / img.with_suffix('.txt').name
    if label.exists():
        return label

    label = img.parent / 'labels' / img.with_suffix('.txt').name
    if label.exists():
        return label

    return None


def create_pseudo_label(image_path, model, conf_threshold=0.5):
    """Create YOLO-format pseudo-label using pre-trained model."""
    frame = cv2.imread(image_path)
    if frame is None:
        return None

    results = model(frame, verbose=False)
    labels = []

    for result in results:
        for box in result.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])

            if cls in COCO_TO_UNIFIED and conf >= conf_threshold:
                x, y, w, h = box.xywhn[0].tolist()
                unified_cls = COCO_TO_UNIFIED[cls]
                labels.append(f"{unified_cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

    return labels if labels else None


def process_dataset_with_labels(dataset_name, data, output_path, max_images):
    """Process dataset that has YOLO labels."""
    stats = {'train': 0, 'val': 0, 'test': 0, 'skipped': 0}
    images = data['images'][:max_images]

    for img_path in tqdm(images, desc=f"{dataset_name} (with labels)"):
        label_path = find_yolo_label(img_path)

        if label_path is None:
            stats['skipped'] += 1
            continue

        split = get_split()
        img_name = f"{dataset_name}_{Path(img_path).name}"
        label_name = img_name.rsplit('.', 1)[0] + '.txt'

        shutil.copy(img_path, f"{output_path}/images/{split}/{img_name}")
        shutil.copy(label_path, f"{output_path}/labels/{split}/{label_name}")
        stats[split] += 1

    return stats


def process_dataset_pseudo(dataset_name, data, output_path, max_images, model):
    """Process dataset without labels using pseudo-labeling."""
    stats = {'train': 0, 'val': 0, 'test': 0, 'skipped': 0}
    images = data['images'][:max_images]

    for img_path in tqdm(images, desc=f"{dataset_name} (pseudo-label)"):
        labels = create_pseudo_label(img_path, model)

        if labels is None:
            stats['skipped'] += 1
            continue

        split = get_split()
        img_name = f"{dataset_name}_{Path(img_path).name}"
        label_name = img_name.rsplit('.', 1)[0] + '.txt'

        shutil.copy(img_path, f"{output_path}/images/{split}/{img_name}")

        with open(f"{output_path}/labels/{split}/{label_name}", 'w') as f:
            f.write('\n'.join(labels))

        stats[split] += 1

    return stats


def extract_video_frames(dataset_name, data, output_path, max_frames_total, model):
    """Extract frames from videos and create pseudo-labels."""
    stats = {'train': 0, 'val': 0, 'test': 0, 'skipped': 0}
    frame_count = 0

    for video_path in tqdm(data['videos'], desc=f"{dataset_name} (videos)"):
        if frame_count >= max_frames_total:
            break

        cap = cv2.VideoCapture(video_path)
        video_name = Path(video_path).stem
        idx = 0

        while cap.isOpened() and frame_count < max_frames_total:
            ret, frame = cap.read()
            if not ret:
                break

            if idx % FRAME_SAMPLE_INTERVAL == 0:
                results = model(frame, verbose=False)
                labels = []

                for result in results:
                    for box in result.boxes:
                        cls = int(box.cls[0])
                        conf = float(box.conf[0])

                        if cls in COCO_TO_UNIFIED and conf >= 0.5:
                            x, y, w, h = box.xywhn[0].tolist()
                            unified_cls = COCO_TO_UNIFIED[cls]
                            labels.append(f"{unified_cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

                if labels:
                    split = get_split()
                    img_name = f"{dataset_name}_{video_name}_{idx:05d}.jpg"
                    label_name = img_name.replace('.jpg', '.txt')

                    cv2.imwrite(f"{output_path}/images/{split}/{img_name}", frame)
                    with open(f"{output_path}/labels/{split}/{label_name}", 'w') as f:
                        f.write('\n'.join(labels))

                    stats[split] += 1
                    frame_count += 1
                else:
                    stats['skipped'] += 1

            idx += 1

        cap.release()

    return stats

In [14]:
def extract_video_frames(dataset_name, data, output_path, max_frames_total, model):
    """Extract frames from videos and create pseudo-labels."""
    VEHICLE_CLASSES = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
    images_dir = Path(output_path) / 'images'
    labels_dir = Path(output_path) / 'labels'

    # Fix: handle None
    max_frames_total = max_frames_total if max_frames_total is not None else float('inf')

    frame_count = 0

    for video_path in tqdm(data['videos'], desc=f"{dataset_name} (videos)"):
        if frame_count >= max_frames_total:
            break

        cap = cv2.VideoCapture(video_path)
        video_name = Path(video_path).stem
        frame_idx = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % 30 == 0:  # Sample every 30 frames
                results = model(frame, verbose=False)
                labels = []

                for result in results:
                    for box in result.boxes:
                        cls = int(box.cls[0])
                        conf = float(box.conf[0])
                        if cls in VEHICLE_CLASSES and conf >= 0.5:
                            x, y, w, h = box.xywhn[0].tolist()
                            class_map = {2: 0, 3: 1, 5: 2, 7: 3}
                            labels.append(f"{class_map[cls]} {x} {y} {w} {h}")

                if labels:
                    split = 'val' if random.random() < 0.2 else 'train'
                    img_name = f"{dataset_name}_{video_name}_{frame_idx:04d}.jpg"
                    cv2.imwrite(str(images_dir / split / img_name), frame)

                    label_name = img_name.replace('.jpg', '.txt')
                    with open(labels_dir / split / label_name, 'w') as f:
                        f.write('\n'.join(labels))
                    frame_count += 1

                    if frame_count >= max_frames_total:
                        break

            frame_idx += 1
        cap.release()

    return {'count': frame_count, 'type': 'pseudo-labeled'}

## 6. Process All Datasets

⏱️ **Estimated time:** 30min - 2hrs depending on data size

In [15]:
all_stats = {}
metadata = {
    'created': datetime.now().isoformat(),
    'license': 'CC BY-NC-SA 4.0',
    'license_note': 'Non-Commercial due to LISA Traffic Light dataset',
    'sources': {},
    'classes': CLASSES,
    'splits': {},
}

print("🚀 Processing datasets...\n")
print("="*60)

for name, data in discovered.items():
    print(f"\n📦 Processing: {name}")
    print("-"*40)

    if data['has_annotations'] and data['images']:
        stats = process_dataset_with_labels(name, data, OUTPUT_PATH, MAX_IMAGES_PER_DATASET)
    elif data['images']:
        stats = process_dataset_pseudo(name, data, OUTPUT_PATH, MAX_IMAGES_PER_DATASET, model)
    elif data['videos']:
        stats = extract_video_frames(name, data, OUTPUT_PATH, MAX_IMAGES_PER_DATASET, model)
    else:
        print(f"⚠️ No processable data found")
        continue

    all_stats[name] = stats
    total = stats['train'] + stats['val'] + stats['test']
    print(f"✅ {name}: {total} samples")

    metadata['sources'][name] = {
        'samples': total,
        'path': data['path'],
    }

total_train = sum(s['train'] for s in all_stats.values())
total_val = sum(s['val'] for s in all_stats.values())
total_test = sum(s['test'] for s in all_stats.values())
total_all = total_train + total_val + total_test

metadata['splits'] = {'train': total_train, 'val': total_val, 'test': total_test}

print("\n" + "="*60)
print(f"📊 TOTAL: {total_all} samples")
print(f"   Train: {total_train}")
print(f"   Val:   {total_val}")
print(f"   Test:  {total_test}")
print("="*60)

🚀 Processing datasets...


📦 Processing: ua-detrac
----------------------------------------


ua-detrac (with labels): 100%|██████████| 13452/13452 [00:12<00:00, 1061.30it/s]


✅ ua-detrac: 0 samples

📦 Processing: real-time-traffic
----------------------------------------


real-time-traffic (videos): 100%|██████████| 2/2 [34:46<00:00, 1043.06s/it]


KeyError: 'train'

## 7. Create Dataset Files

In [ ]:
with open(f'{OUTPUT_PATH}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

yaml_content = f"""# TMS2 Unified Traffic Dataset
# License: CC BY-NC-SA 4.0 (Non-Commercial)
# Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}

path: {OUTPUT_PATH}
train: images/train
val: images/val
test: images/test

names:
  0: car
  1: motorcycle
  2: bus
  3: truck
  4: traffic_light
"""

with open(f'{OUTPUT_PATH}/dataset.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ Created dataset.yaml")
print("✅ Created metadata.json")

In [ ]:
readme_content = f"""# TMS2 Unified Traffic Dataset

A unified traffic detection dataset for vehicle and traffic light detection.

## ⚠️ License: CC BY-NC-SA 4.0

**This dataset is NON-COMMERCIAL** due to the inclusion of the LISA Traffic Light dataset.

## 📊 Statistics

| Split | Images |
|-------|--------|
| Train | {total_train:,} |
| Val | {total_val:,} |
| Test | {total_test:,} |
| **Total** | **{total_all:,}** |

## 🏷️ Classes

| ID | Class |
|----|-------|
| 0 | car |
| 1 | motorcycle |
| 2 | bus |
| 3 | truck |
| 4 | traffic_light |

## 🙏 Source Credits

- UA-DETRAC (Beijing Institute of Technology)
- Real-Time Traffic Dataset (unidpro/Kaggle)
- Serbia Traffic (unidpro/Kaggle)
- Traffic Density Singapore (rahat52/Kaggle)
- LISA Traffic Light (UC San Diego / BEUMER Group)

## 🚀 Usage

```python
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='dataset.yaml', epochs=50)
```

---
Generated: {datetime.now().strftime('%Y-%m-%d')} | Author: {KAGGLE_USERNAME}
"""

with open(f'{OUTPUT_PATH}/README.md', 'w') as f:
    f.write(readme_content)

print("✅ Created README.md")

## 8. Setup Kaggle API

In [ ]:
from google.colab import files

print("📤 Please upload your kaggle.json file:")
print("   (Download from: kaggle.com -> Account -> API -> Create New Token)\n")

try:
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print("\n✅ Kaggle API configured!")
except:
    print("⚠️ No file uploaded.")

## 9. Create Kaggle Metadata & Upload

In [ ]:
kaggle_metadata = {
    "title": DATASET_TITLE,
    "id": f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
    "licenses": [{"name": "CC-BY-NC-SA-4.0"}],
    "keywords": ["traffic", "vehicles", "object detection", "yolo", "traffic lights"]
}

with open(f'{OUTPUT_PATH}/dataset-metadata.json', 'w') as f:
    json.dump(kaggle_metadata, f, indent=2)

print("✅ Created dataset-metadata.json")
print(f"   License: CC BY-NC-SA 4.0 (Non-Commercial)")

In [ ]:
import subprocess

result = subprocess.run(
    ['kaggle', 'datasets', 'list', '-m', '--user', KAGGLE_USERNAME],
    capture_output=True, text=True
)

dataset_exists = DATASET_SLUG in result.stdout
print(f"{'📦 Updating existing' if dataset_exists else '🆕 Creating new'} dataset...")

In [ ]:
print(f"🚀 Uploading to Kaggle...\n")

if dataset_exists:
    !kaggle datasets version -p "{OUTPUT_PATH}" -m "Updated with {total_all} images"
else:
    !kaggle datasets create -p "{OUTPUT_PATH}" --public

print(f"\n✅ Done! View at: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_SLUG}")

## ✅ Complete!

Your unified dataset (with LISA) is now on Kaggle.

**Remember:** This dataset is **CC BY-NC-SA 4.0** (Non-Commercial).

```python
!kaggle datasets download kunlaisnghh25/tms2-traffic-unified
```